# Resampling: Decimation, Interpolation and Rate Conversion
Changing the sample rate is filtering plus bookkeeping:
1. **Decimation** (lower the rate by an integer factor $M$): low-pass filter, then keep every $M$-th sample.
2. **Interpolation** (raise the rate by an integer factor $L$): insert $L - 1$ zeros between samples, then low-pass filter.
3. **Rational conversion** by $L/M$ (for example 44.1 to 48 kHz): interpolate by $L$ and decimate by $M$ with one shared filter, computed efficiently by a **polyphase** filter.

`00_Audio_Signals/sampling_and_quantization.ipynb` shows what aliasing sounds like. This notebook is about the filters that prevent it.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from audiodsp import multi_sine, spectrum_db

%matplotlib inline
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 5)

def level_db(x, f, fs):
    """Level in dB (re full-scale sine) of the spectral component at frequency f.

    Uses a whole number of seconds and integer frequencies, so every tone falls exactly on a bin.
    """
    X = np.fft.rfft(x)
    return 20 * np.log10(np.abs(X[int(round(f * len(x) / fs))]) / (len(x) / 2) + 1e-12)

## 1. Decimation
Lowering the rate from $f_s$ to $f_s/M$ moves the Nyquist frequency down to $f_s/(2M)$. Anything above it must be removed first, or it aliases.

The test signal at 48 kHz is a 1 kHz tone plus a 5.5 kHz tone, decimated by 6 to 8 kHz. 5.5 kHz is above the new 4 kHz Nyquist frequency, so it should disappear. Without a filter it aliases to 2.5 kHz. The tones are chosen so the alias lands where nothing else is: an alias that lands exactly on an existing tone can cancel it or add to it, which hides what happened.

`scipy.signal.resample_poly(x, 1, M)` applies a Kaiser-windowed FIR anti-alias filter and decimates in one step.

In [ ]:
fs_in, M = 48000, 6
fs_out = fs_in // M
x = multi_sine([1000, 5500], 1.0, fs_in, amplitudes=[0.5, 0.5])

naive = x[::M]
filtered = signal.resample_poly(x, 1, M)

fig, ax = plt.subplots(figsize=(12, 4))
for y, label in [(naive, 'keep every 6th sample'), (filtered, 'resample_poly (filter, then keep every 6th)')]:
    f, mag = spectrum_db(y, fs_out)
    ax.plot(f, mag, label=label)
ax.set_ylim(-120, 5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('dB')
ax.set_title('Decimating 48 kHz to 8 kHz')
ax.legend()
plt.show()
for y, label in [(naive, 'no filter'), (filtered, 'resample_poly')]:
    print(f'{label:<14} 1 kHz tone: {level_db(y, 1000, fs_out):7.1f} dB   2.5 kHz alias: {level_db(y, 2500, fs_out):7.1f} dB')

## 2. Interpolation
Raising the rate by $L$ starts by inserting $L - 1$ zeros after every sample (**zero-stuffing**). The samples that were there are unchanged, so the spectrum below the old Nyquist frequency is unchanged too. But the new, higher Nyquist frequency exposes the **images** of the spectrum around multiples of the old rate, the same images seen in `00_Audio_Signals`. An **anti-imaging** low-pass filter at the old Nyquist frequency removes them and fills in the zeros with interpolated values. Zero-stuffing also spreads the signal's energy over $L$ times as many samples, so the filter has a gain of $L$.

In [ ]:
fs_low, L = 8000, 4
fs_high = fs_low * L
x_low = multi_sine([1000, 3000], 1.0, fs_low, amplitudes=[0.5, 0.3])

stuffed = np.zeros(len(x_low) * L)
stuffed[::L] = x_low
anti_imaging = L * signal.firwin(8 * L * 4 + 1, fs_low / 2, fs=fs_high)
# lfilter would delay the output by (taps - 1) / 2 samples; 'same' convolution keeps it aligned for plotting
interpolated = np.convolve(stuffed, anti_imaging, mode='same')

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for y, fs_y, label in [(stuffed, fs_high, 'zero-stuffed'), (interpolated, fs_high, 'after anti-imaging filter')]:
    f, mag = spectrum_db(y, fs_y)
    axes[0].plot(f / 1000, mag, label=label)
for k in range(1, L):
    axes[0].axvline(k * fs_low / 1000, color='k', linestyle=':', linewidth=1)
axes[0].set_ylim(-120, 5)
axes[0].set_xlabel('Frequency (kHz)   (dotted: multiples of the old 8 kHz rate)')
axes[0].set_title(f'Spectrum at {fs_high // 1000} kHz')
axes[0].legend(loc='lower left')
view = slice(400, 440)
axes[1].plot(np.arange(len(stuffed))[view], stuffed[view], 'o', label='zero-stuffed')
axes[1].plot(np.arange(len(stuffed))[view], interpolated[view], '.-', label='interpolated')
axes[1].set_xlabel(f'Sample index at {fs_high // 1000} kHz')
axes[1].set_title('Time domain: the filter fills in the zeros')
axes[1].legend()
plt.tight_layout()
plt.show()

middle = slice(len(interpolated) // 4, 3 * len(interpolated) // 4)
ideal = multi_sine([1000, 3000], 1.0, fs_high, amplitudes=[0.5, 0.3])
error = interpolated[middle] - ideal[middle]
print(f'Interpolation error vs the ideal {fs_high // 1000} kHz signal: {10 * np.log10(np.mean(ideal[middle]**2) / np.mean(error**2)):.1f} dB SNR')

## 3. Rational conversion: 44.1 kHz to 48 kHz
$48000 / 44100 = 160 / 147$. Done literally, the conversion would zero-stuff by 160 to an intermediate rate of 7.056 MHz, filter there, and keep every 147th sample. Almost all of that work is wasted: the inputs to the filter are mostly zeros, and almost all of its outputs are thrown away.

A **polyphase** filter splits the anti-aliasing filter into $L$ short sub-filters and computes only the output samples that are kept, using only the input samples that are not zero. `resample_poly(x, 160, 147)` does exactly this.

The table converts single tones and measures the result against tones generated directly at 48 kHz. The filter's transition band sits just below the Nyquist frequency, so tones close to it are attenuated.

In [ ]:
L, M = 160, 147
fs_a, fs_b = 44100, 48000
duration = 1.0

print(f'{"Tone":>8}{"Gain":>10}{"SNR vs ideal":>15}')
for tone in [1000, 10000, 15000, 19000, 20000, 21000]:
    x_a = np.sin(2 * np.pi * tone * np.arange(int(duration * fs_a)) / fs_a)
    x_b = signal.resample_poly(x_a, L, M)
    ideal = np.sin(2 * np.pi * tone * np.arange(len(x_b)) / fs_b)
    middle = slice(len(x_b) // 4, 3 * len(x_b) // 4)
    gain = np.sqrt(np.mean(x_b[middle]**2) / np.mean(ideal[middle]**2))
    snr = 10 * np.log10(np.mean(ideal[middle]**2) / np.mean((x_b[middle] - ideal[middle])**2))
    print(f'{tone:>6} Hz{20 * np.log10(gain):>7.2f} dB{snr:>12.1f} dB')

x_a = np.random.default_rng(0).normal(0, 0.1, 10 * fs_a)
t0 = time.perf_counter()
signal.resample_poly(x_a, L, M)
print(f'\nConverting 10 s of audio with resample_poly took {1000 * (time.perf_counter() - t0):.0f} ms')

## 4. Where resampling is used
- **Sample-rate conversion** between 44.1 kHz (CD) and 48 kHz (video, most audio interfaces).
- **Oversampling** for nonlinear effects: saturation creates harmonics above the Nyquist frequency, which alias unless the effect runs at a higher rate (`05_Audio_Effects`).
- **Oversampling converters**: ADCs and DACs run at many times the audio rate so their analog filters can be simple, and do the sharp filtering digitally.
- **Pitch shifting and time stretching** (`05_Audio_Effects`), **filter banks in codecs** (`06_Psychoacoustics_and_Audio_Coding`), and **telephone audio** at 8 kHz (`09_Capstone_Projects`).